# Práctica 1 – Búsqueda de imágenes satélite

## Este script permite buscar imágenes Sentinel-2 sobre Cataluña en los últimos 30 días, seleccionar las escenas con menor cobertura de nubes y generar un archivo GeoJSON con los footprints.

#### Importación de librerías

En este bloque se importan las librerías necesarias para realizar la búsqueda de imágenes, gestionar fechas y trabajar con datos en formato JSON.

In [13]:
import requests
import json
from datetime import date, timedelta

#### Definición del rango temporal

En este bloque se calcula automáticamente el periodo de búsqueda: desde la fecha actual hasta 30 días antes.

In [14]:
today = date.today()
start_date = today - timedelta(days=30)

print("Hoy:", today)
print("Hace 30 días:", start_date)

Hoy: 2026-04-27
Hace 30 días: 2026-03-28


#### Definición del área de estudio

En este bloque se define el bounding box correspondiente al territorio de Cataluña, que se utilizará para la búsqueda de imágenes.

In [15]:
# Bounding box de Cataluña (WGS84)
bbox = [0.1500, 40.5200, 3.3300, 42.8620]

print("Bounding box:", bbox)

Bounding box: [0.15, 40.52, 3.33, 42.862]


#### Búsqueda de imágenes Sentinel-2

En este bloque se realiza la consulta a la API STAC de Copernicus para obtener las imágenes disponibles en el área de estudio y el rango temporal definido.

In [16]:
url = "https://stac.dataspace.copernicus.eu/v1/search"

params = {
    "collections": "sentinel-2-l2a",
    "bbox": ",".join(map(str, bbox)),
    "datetime": f"{start_date}T00:00:00Z/{today}T23:59:59Z",
    "limit": 100
}

response = requests.get(url, params=params)

print("Status:", response.status_code)

Status: 200


####  Procesamiento de resultados

En este bloque se procesan los resultados obtenidos de la API, extrayendo el número total de escenas disponibles.

In [17]:
data = response.json()
features = data.get("features", [])

print("Escenas encontradas:", len(features))

Escenas encontradas: 100


#### Extracción de información relevante

En este bloque se extraen los datos clave de cada escena, como el identificador, la fecha de adquisición y el porcentaje de nubes.

In [18]:
for item in features[:5]:
    print("ID:", item["id"])
    print("Fecha:", item["properties"]["datetime"])
    print("Nubes:", item["properties"]["eo:cloud_cover"])
    print("----")

ID: S2C_MSIL2A_20260426T104021_N0512_R008_T31TEH_20260426T160712
Fecha: 2026-04-26T10:40:21.025000Z
Nubes: 20.27
----
ID: S2C_MSIL2A_20260426T104021_N0512_R008_T31TEG_20260426T160712
Fecha: 2026-04-26T10:40:21.025000Z
Nubes: 12.74
----
ID: S2C_MSIL2A_20260426T104021_N0512_R008_T31TEF_20260426T160712
Fecha: 2026-04-26T10:40:21.025000Z
Nubes: 5.8
----
ID: S2C_MSIL2A_20260426T104021_N0512_R008_T31TEE_20260426T160712
Fecha: 2026-04-26T10:40:21.025000Z
Nubes: 20.91
----
ID: S2C_MSIL2A_20260426T104021_N0512_R008_T31TDH_20260426T160712
Fecha: 2026-04-26T10:40:21.025000Z
Nubes: 34.33
----


#### Agrupación por footprint (tile)

En este bloque se agrupan las escenas en función de la tile a la que pertenecen, con el objetivo de seleccionar una única escena por cada footprint.

In [19]:
def get_tile(item):
    return item["id"].split("_")[5]

In [20]:
tiles = {}

for item in features:
    tile = get_tile(item)  # usamos la función
    cloud = item["properties"]["eo:cloud_cover"]

    if tile not in tiles:
        tiles[tile] = item
    else:
        if cloud < tiles[tile]["properties"]["eo:cloud_cover"]:
            tiles[tile] = item

print("Número de tiles seleccionadas:", len(tiles))

Número de tiles seleccionadas: 19


#### Selección de la mejor escena por tile

En este bloque se obtiene, para cada tile, la escena con menor porcentaje de nubes y se genera un listado con su identificador, fecha de adquisición y sensor.

In [21]:
for tile in sorted(tiles.keys()):
    item = tiles[tile]
    print("Tile:", tile)
    print("ID:", item["id"])
    print("Fecha:", item["properties"]["datetime"])
    print("Nubes:", item["properties"]["eo:cloud_cover"])
    print("Sensor:", item["id"].split("_")[0])
    print("----")

Tile: T30TYK
ID: S2C_MSIL2A_20260426T104021_N0512_R008_T30TYK_20260426T160712
Fecha: 2026-04-26T10:40:21.025000Z
Nubes: 0.3
Sensor: S2C
----
Tile: T30TYL
ID: S2C_MSIL2A_20260419T105021_N0512_R051_T30TYL_20260419T161518
Fecha: 2026-04-19T10:50:21.025000Z
Nubes: 4.1
Sensor: S2C
----
Tile: T30TYM
ID: S2A_MSIL2A_20260421T105051_N0512_R051_T30TYM_20260421T185511
Fecha: 2026-04-21T10:50:51.024000Z
Nubes: 19.66
Sensor: S2A
----
Tile: T30TYN
ID: S2B_MSIL2A_20260424T104619_N0512_R051_T30TYN_20260424T132142
Fecha: 2026-04-24T10:46:19.024000Z
Nubes: 25.9
Sensor: S2B
----
Tile: T31TBE
ID: S2C_MSIL2A_20260426T104021_N0512_R008_T31TBE_20260426T160712
Fecha: 2026-04-26T10:40:21.025000Z
Nubes: 0.28
Sensor: S2C
----
Tile: T31TBF
ID: S2C_MSIL2A_20260419T105021_N0512_R051_T31TBF_20260419T161518
Fecha: 2026-04-19T10:50:21.025000Z
Nubes: 3.93
Sensor: S2C
----
Tile: T31TBG
ID: S2A_MSIL2A_20260421T105051_N0512_R051_T31TBG_20260421T185511
Fecha: 2026-04-21T10:50:51.024000Z
Nubes: 19.39
Sensor: S2A
----
Tile: 

#### Generación del archivo GeoJSON

En este bloque se genera un archivo GeoJSON con la geometría de los footprints seleccionados (una por cada tile).

In [10]:
def save_geojson(tiles, filename):
    geojson = {
        "type": "FeatureCollection",
        "features": []
    }

    for tile, item in tiles.items():
        feature = {
            "type": "Feature",
            "geometry": item["geometry"],
            "properties": {
                "id": item["id"],
                "fecha": item["properties"]["datetime"],
                "nubes": item["properties"]["eo:cloud_cover"]
            }
        }
        geojson["features"].append(feature)

    with open(filename, "w") as f:
        json.dump(geojson, f, indent=2)

In [11]:
save_geojson(tiles, "footprints.geojson")

print("GeoJSON generado correctamente")

GeoJSON generado correctamente
